# Part 1:  Problem Definition & Dataset Verification

# Task 1

In [2]:
import pandas as pd
df=pd.read_csv("../LAB 03/clean_churn.csv")
df.duplicated().sum()
df = df.drop_duplicates() # forgot to save drop_duplicated back in csv in prev labs 
df.to_csv("../LAB 03/clean_churn.csv", index=False)
print("shape of dataset :",df.shape)
print("data types of dataset :\n",df.dtypes)

shape of dataset : (7021, 20)
data types of dataset :
 gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                   str
dtype: object


# Task 1 interpretation
The cleaned Telco Churn dataset contains 7,021 customer records and 20 columns. The dataset contains both numerical and categorical variables. The numerical variables include tenure, SeniorCitizen, MonthlyCharges, and TotalCharges, while variables such as Contract, InternetService, and PaymentMethod are categorical. The target variable is Churn, which represents whether a customer left the service.

# Task 2
In Lab 3, the Decision Tree achieved an accuracy of 79.43%, precision of 62.03%, recall of 57.53%, and F1-score of 59.69% on the test set. In Lab 4, the default Random Forest achieved an accuracy of 77.22%, while the tuned Random Forest achieved 79.86% accuracy, 65.61% precision, 50.27% recall, and 56.93% F1-score. These results will be used as benchmarks when comparing Logistic Regression and KNN in this lab.

# Part 2 : Feature Scaling & Preparation 

# Task 3

In [3]:
X=df.drop("Churn",axis=1)
y=df["Churn"]
y = y.map({"No": 0, "Yes": 1})
print(y.value_counts())
X = pd.get_dummies(X, drop_first=True)
print("X shape after encoding:", X.shape)
print("\nData types:")
print(X.dtypes)

Churn
0    5164
1    1857
Name: count, dtype: int64
X shape after encoding: (7021, 30)

Data types:
SeniorCitizen                              int64
tenure                                     int64
MonthlyCharges                           float64
TotalCharges                             float64
gender_Male                                 bool
Partner_Yes                                 bool
Dependents_Yes                              bool
PhoneService_Yes                            bool
MultipleLines_No phone service              bool
MultipleLines_Yes                           bool
InternetService_Fiber optic                 bool
InternetService_No                          bool
OnlineSecurity_No internet service          bool
OnlineSecurity_Yes                          bool
OnlineBackup_No internet service            bool
OnlineBackup_Yes                            bool
DeviceProtection_No internet service        bool
DeviceProtection_Yes                        bool
TechSupport_No int

# Task 3 interpretation
The dataset was divided into features (X) and the target variable (y). The Churn target was converted from categorical values into binary values, where 0 represents “No” churn and 1 represents “Yes” churn. The categorical input features were converted into numerical form using one-hot encoding so that they could be used by machine-learning algorithms. The increase in the number of columns after encoding is expected because categorical variables are represented using separate numerical indicator columns.

# Task 4
Feature scaling is important for KNN because KNN uses distance calculations, so features with larger numerical ranges could dominate the distance. Scaling is also helpful for Logistic Regression because it puts features on comparable scales and can improve numerical optimization. Decision Trees and Random Forests do not require scaling because they make decisions using feature-based split thresholds rather than distance calculations.

# Task 5


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing  import StandardScaler
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)
scaler=StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Scaled training data shape:", X_train_scaled.shape)
print("Scaled test data shape:", X_test_scaled.shape)

X_train: (5616, 30)
X_test: (1405, 30)
y_train: (5616,)
y_test: (1405,)
Scaled training data shape: (5616, 30)
Scaled test data shape: (1405, 30)


# Task 5 interpretation
The dataset was divided into 80% training data and 20% testing data using a fixed random state of 42. Stratified splitting was used to maintain a similar proportion of churn and non-churn customers in both sets. StandardScaler was fitted only on the training data and then used to transform both the training and testing data. This prevents data leakage because information from the test set is not used to learn the scaling parameters. The scaled data will be used for Logistic Regression and KNN, while the previously used Decision Tree and Random Forest models do not require feature scaling.

# Part 3: Logistic Regression 

# Task 6


In [5]:
from sklearn.linear_model import LogisticRegression 
lr_model=LogisticRegression(random_state=42)
lr_model.fit(X_train_scaled,y_train)
y_pred_lr=lr_model.predict(X_test_scaled)
print("Logistic Regression predictions:", y_pred_lr[:10])

Logistic Regression predictions: [1 1 0 0 0 0 0 1 0 0]


# Task 6 interpretation
A Logistic Regression model was trained using the scaled training data. The trained model was then used to predict churn on the unseen test data. The predictions contain binary class labels, where 0 represents no churn and 1 represents churn. These predictions will be evaluated using accuracy, precision, recall, F1-score, and a confusion matrix later.

# Task 7

In [8]:
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,confusion_matrix)
accuracy=accuracy_score(y_test,y_pred_lr)
precision=precision_score(y_test,y_pred_lr)
recall=recall_score(y_test,y_pred_lr)
f1=f1_score(y_test,y_pred_lr)
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1-score :", f1)

cm=confusion_matrix(y_test,y_pred_lr)
print("\nConfusion Matrix:")
print(cm)

Accuracy : 0.80355871886121
Precision: 0.6621621621621622
Recall   : 0.5268817204301075
F1-score : 0.5868263473053892

Confusion Matrix:
[[933 100]
 [176 196]]


# Task 7 interpretation
The Logistic Regression model achieved an accuracy of 80.36%, meaning that it correctly classified most of the test customers. Its precision was 66.22%, indicating that about two-thirds of the customers predicted as churners actually churned. The recall was 52.69%, meaning that the model correctly detected approximately half of the customers who actually churned. The F1-score was 58.68%, reflecting the balance between precision and recall. The confusion matrix shows 933 true negatives, 100 false positives, 176 false negatives, and 196 true positives. In particular, the 176 false negatives show that the model missed a considerable number of customers who actually churned.

# Task 8

In [10]:
y_prob_lr = lr_model.predict_proba(X_test_scaled)
print("First 10 probability predictions:")
print(y_prob_lr[:10,1])

First 10 probability predictions:
[0.75891304 0.6147742  0.28394934 0.0486018  0.00902546 0.32980849
 0.15474878 0.79230857 0.01749746 0.01511708]


# Task 8 interpretation
The predict_proba() output provides the estimated probability of churn for each test customer. For example, the first customer has a churn probability of approximately 75.89%, while the third customer has a probability of approximately 28.39%. Using the default threshold of 0.5, the first customer would be classified as a churner, while the third would be classified as a non-churner. Therefore, probability predictions provide more information about the model's confidence than the binary predictions returned by predict().
